#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.window import Window
from pyspark.sql.functions import trim, col

#Read Bronze Table


In [0]:
df = spark.table("demo_project.bronze.crm_prd_info")

#Analyzing Data

In [0]:
df.limit(10).display()

#Silver Transformations

##Renaming Columns

In [0]:
rename_map = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity checks

In [0]:
df.limit(10).display()

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name,trim(col(field.name)))

In [0]:
df = df.withColumn("product_line",
                   F.when(F.upper(col("product_line")) == "M", "Mountain")
                   .when(F.upper(col("product_line")) == "T", "Touring")
                   .when(F.upper(col("product_line")) == "R", "Road")
                   .when(F.upper(col("product_line")) == "S", "Other Sales")
                   .otherwise("N/A"))

##Product Key Parsing

In [0]:


df = df.withColumn("category_id", F.regexp_replace(F.substring(col("product_key"), 1, 5), "-", "_"))
df = df.withColumn("product_key", F.substring(col("product_key"), 7, F.length(col("product_key"))))

##Date Casting

In [0]:
df = df.withColumn("start_date",
                   col("start_date").cast(DateType()))

##Cost Cleanup

In [0]:
df = df.withColumn("product_cost",
                   F.coalesce(col("product_cost"),F.lit(0)))

##Sanity checks

In [0]:
df.limit(10).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").saveAsTable("demo_project.silver.crm_products")

##Sanity checks of Silver Table

In [0]:
%sql
select * from limit 10;